# 7. Retrieval-Augmented Generation (RAG)

RAG gives a model access to information it wasn't trained on: split a document
collection into chunks, embed each chunk, store the embeddings in a vector store,
then for a given question retrieve the most similar chunks and stuff them into
the prompt before asking the model to answer.

This notebook uses the in-memory vector store — the simplest of the three
backends the app supports (Redis and Postgres/pgvector are the other two; same
pipeline, durable storage — see `docs/langchain/07-rag/`). It reuses the
project's sample corpus (`langchain_demo/rag_corpus.md`) so the notebook and the
app demo the exact same document collection.

**Prerequisites:** Ollama running locally with `llama3.2` and `qwen3-embedding:0.6b`
pulled (`ollama pull qwen3-embedding:0.6b`).

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

## Load and chunk the corpus

In [ ]:
import uuid

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

corpus_path = project_root / "langchain_demo" / "rag_corpus.md"
text = corpus_path.read_text()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)
# Stable, content-derived ids so re-running this cell upserts instead of duplicating.
documents = [Document(page_content=chunk, id=str(uuid.uuid5(uuid.NAMESPACE_URL, chunk))) for chunk in chunks]
print(f"{len(documents)} chunks")
print(documents[0].page_content[:200])

## Embed and store

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

from models.embedding_models.ollama_models import qwen3_embedding_model

store = InMemoryVectorStore(embedding=qwen3_embedding_model)
store.add_documents(documents)
print("ingested", len(documents), "chunks")

## Retrieve + answer

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

from models.chat_models.ollama_models import SupportedModel, get_chat_model

question = "What is RAG used for?"
retrieved = store.similarity_search(question, k=4)
for doc in retrieved:
    print("-", doc.page_content[:100].replace("\n", " "), "...")

In [ ]:
ANSWER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using ONLY the provided context. If the context "
            "doesn't contain the answer, say so.\n\nContext:\n{context}",
        ),
        ("human", "{question}"),
    ]
)

llm = get_chat_model(SupportedModel.llama3_2)
context = "\n\n".join(doc.page_content for doc in retrieved)
chain = ANSWER_PROMPT | llm
answer = chain.invoke({"context": context, "question": question})
print(answer.content)

## 🧪 Playground

**1. A different question** — try `"How does LangGraph differ from LangChain?"` or `"What vector store backends does this project support?"`.

In [ ]:
# TODO: retrieve + answer a different question


**2. Change `k`** (retrieve more or fewer chunks) and see how the answer's grounding changes.

In [ ]:
# TODO: retrieve with k=1 vs k=8 and compare the answers


**3. Ask something NOT in the corpus** (e.g. `"What's the capital of France?"`) — does the model correctly say the context doesn't contain the answer, or does it answer from its own knowledge anyway?

In [ ]:
# TODO: ask an out-of-corpus question and inspect the answer
